# GramSathi - Data Exploration

Exploring synthetic citizen data for the scheme eligibility prediction model.

**Objectives:**
- Load and understand the dataset
- Analyze distributions by state, caste, income, occupation
- Correlation analysis
- Missing data patterns
- Feature engineering ideas
- Baseline eligibility distributions

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='husl')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

print('Libraries loaded successfully')

In [ ]:
import sys
sys.path.append('..')
from trainer.data_pipeline import (
    generate_synthetic_citizen_data,
    compute_eligibility_labels,
    TARGET_COLUMNS
)

df = generate_synthetic_citizen_data(n_samples=100000, seed=42)
labels = compute_eligibility_labels(df)
df_full = pd.concat([df, labels], axis=1)
print(f'Dataset shape: {df_full.shape}')
df_full.head()

## 1. Basic Statistics

In [ ]:
df.describe(include='all')

In [ ]:
print('Missing values per column:')
print(df.isnull().sum()[df.isnull().sum() > 0])

## 2. Distribution Analysis

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 10))

state_counts = df['state'].value_counts()
axes[0, 0].barh(state_counts.index, state_counts.values, color='steelblue')
axes[0, 0].set_title('Citizens by State')
axes[0, 0].set_xlabel('Count')

caste_counts = df['caste_category'].value_counts()
axes[0, 1].bar(caste_counts.index, caste_counts.values, color=['gold', 'lightcoral', 'lightgreen', 'lightskyblue'])
axes[0, 1].set_title('Caste Distribution')
axes[0, 1].set_xlabel('Category')
axes[0, 1].tick_params(axis='x', rotation=0)

axes[0, 2].hist(df['age'], bins=30, color='steelblue', edgecolor='white', alpha=0.7)
axes[0, 2].axvline(df['age'].median(), color='red', linestyle='--', label=f"Median: {df['age'].median():.0f}")
axes[0, 2].set_title('Age Distribution')
axes[0, 2].set_xlabel('Age')
axes[0, 2].legend()

gender_counts = df['gender'].value_counts()
axes[0, 3].pie(gender_counts.values, labels=gender_counts.index, autopct='%1.1f%%',
               colors=['lightcoral', 'lightblue', 'lightgreen'])
axes[0, 3].set_title('Gender Distribution')

occ_counts = df['occupation'].value_counts()
axes[1, 0].barh(occ_counts.index, occ_counts.values, color='teal')
axes[1, 0].set_title('Occupation Distribution')
axes[1, 0].set_xlabel('Count')

axes[1, 1].hist(df['annual_income'] / 100000, bins=40, color='green', edgecolor='white', alpha=0.7)
axes[1, 1].axvline(df['annual_income'].median() / 100000, color='red', linestyle='--',
                   label=f"Median: {df['annual_income'].median()/100000:.1f}L")
axes[1, 1].set_title('Annual Income Distribution')
axes[1, 1].set_xlabel('Annual Income (Lakhs)')
axes[1, 1].legend()

edu_counts = df['education_level'].value_counts()
axes[1, 2].barh(edu_counts.index, edu_counts.values, color='purple')
axes[1, 2].set_title('Education Level Distribution')
axes[1, 2].set_xlabel('Count')

axes[1, 3].hist(df['family_size'], bins=range(1, 16), color='orange', edgecolor='white', alpha=0.7, align='left')
axes[1, 3].set_title('Family Size Distribution')
axes[1, 3].set_xlabel('Family Size')

plt.tight_layout()
plt.show()

## 3. Income Analysis by Demographic Groups

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.boxplot(x='caste_category', y='annual_income', data=df, ax=axes[0], palette='Set2')
axes[0].set_title('Income Distribution by Caste')
axes[0].tick_params(axis='x', rotation=0)

sns.boxplot(x='education_level', y='annual_income', data=df, ax=axes[1], palette='Set3')
axes[1].set_title('Income Distribution by Education')
axes[1].tick_params(axis='x', rotation=45)

sns.violinplot(x='gender', y='annual_income', data=df, ax=axes[2], palette='pastel')
axes[2].set_title('Income Distribution by Gender')

plt.tight_layout()
plt.show()

## 4. Correlation Analysis

In [ ]:
numeric_cols = ['age', 'annual_income', 'land_holding', 'family_size', 'is_farmer', 'has_disability']
corr_matrix = df[numeric_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            square=True, linewidths=0.5)
plt.title('Feature Correlation Matrix', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
eligible_cols = TARGET_COLUMNS
eligible_corr = labels[eligible_cols].corr()

plt.figure(figsize=(10, 8))
sns.heatmap(eligible_corr, annot=True, cmap='YlOrRd', center=0.5, vmin=0, vmax=1,
            square=True, linewidths=0.5, fmt='.2f')
plt.title('Scheme Eligibility Correlation', fontsize=14)
plt.tight_layout()
plt.show()

## 5. Eligibility Distribution

In [ ]:
eligibility_counts = labels.sum().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(range(len(eligibility_counts)), eligibility_counts.values, color='coral')
ax.set_yticks(range(len(eligibility_counts)))
ax.set_yticklabels([c.replace('_', ' ').title() for c in eligibility_counts.index])
ax.set_xlabel('Number of Eligible Citizens')
ax.set_title('Ground Truth Eligibility Distribution Across Schemes')

for bar, val in zip(bars, eligibility_counts.values):
    ax.text(bar.get_width() + 200, bar.get_y() + bar.get_height() / 2,
            f'{val:,} ({100*val/len(df):.1f}%)', va='center', fontsize=10)

plt.tight_layout()
plt.show()

## 6. Caste & Income Impact on Eligibility

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 8))
axes = axes.flatten()

for i, scheme in enumerate(TARGET_COLUMNS):
    scheme_data = df_full[df_full[scheme] == 1]
    if len(scheme_data) == 0:
        axes[i].text(0.5, 0.5, 'No eligible cases', ha='center', va='center')
        continue
    caste_order = ['General', 'OBC', 'SC', 'ST']
    caste_counts = scheme_data['caste_category'].value_counts().reindex(caste_order).fillna(0)
    axes[i].bar(caste_order, caste_counts.values, color=['gold', 'orange', 'lightcoral', 'lightgreen'])
    axes[i].set_title(f"{scheme.replace('_', ' ').title()}", fontsize=10)
    axes[i].tick_params(axis='x', rotation=0)

plt.suptitle('Eligibility Distribution by Caste Category', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 7. Feature Engineering Ideas

Based on the exploration, the following features could improve model performance:

1. **Income to Land Ratio**: `annual_income / (land_holding + 1)`
2. **Dependency Ratio**: `family_size / income_per_capita`
3. **Age Groups**: Binned categories (18-25, 26-35, 36-50, 51-65, 65+)
4. **Education Numeric**: Ordinal encoding of education level
5. **Region Clusters**: Grouping states into zones (North, South, East, West, Central, NE)
6. **Income Category**: Below poverty line, low income, middle income, high income
7. **Farmer with Land**: Interaction feature `is_farmer * land_holding`
8. **Vulnerability Score**: Composite score combining caste, income, disability, age

In [ ]:
df_engineered = df.copy()
df_engineered['income_per_family_member'] = df['annual_income'] / (df['family_size'] + 1)
df_engineered['income_land_ratio'] = df['annual_income'] / (df['land_holding'] + 0.1)
df_engineered['is_senior'] = (df['age'] >= 60).astype(int)
df_engineered['is_youth'] = ((df['age'] >= 18) & (df['age'] <= 35)).astype(int)

region_map = {
    'Jammu & Kashmir': 'North', 'Himachal Pradesh': 'North', 'Punjab': 'North',
    'Uttarakhand': 'North', 'Haryana': 'North', 'Delhi': 'North',
    'Rajasthan': 'West', 'Gujarat': 'West', 'Maharashtra': 'West', 'Goa': 'West',
    'Uttar Pradesh': 'Central', 'Madhya Pradesh': 'Central', 'Chhattisgarh': 'Central',
    'Bihar': 'East', 'West Bengal': 'East', 'Jharkhand': 'East', 'Odisha': 'East',
    'Assam': 'North East', 'Sikkim': 'North East', 'Arunachal Pradesh': 'North East',
    'Nagaland': 'North East', 'Manipur': 'North East', 'Mizoram': 'North East',
    'Tripura': 'North East', 'Meghalaya': 'North East',
    'Karnataka': 'South', 'Andhra Pradesh': 'South', 'Telangana': 'South',
    'Kerala': 'South', 'Tamil Nadu': 'South', 'Puducherry': 'South'
}
df_engineered['region'] = df['state'].map(region_map).fillna('Other')

df_engineered[['income_per_family_member', 'income_land_ratio', 'is_senior', 'is_youth', 'region']].head()

In [ ]:
print('Summary of exploration complete.')
print(f'Total unique citizens: {len(df):,}')
print(f'States represented: {df["state"].nunique()}')
print(f'Districts represented: {df["district"].nunique()}')
print(f'Overall eligibility rate: {100 * labels.values.sum() / (len(df) * len(TARGET_COLUMNS)):.1f}%')